# Train SmallM-Nova from scratch on a free Kaggle GPU

This notebook trains the **brand-new Nova architecture** (recurrent-depth
transformer) from scratch on real TinyStories, then does instruction SFT on
real Dolly-15k, agentic SFT and model soup.

**Before you run:** Settings -> Internet -> **On** (needed to clone the repo
and download the datasets). Then Run all.

Runtime: about 1-2 hours for the 90M preset on a T4; the 30M preset is faster.

In [ ]:
import os, subprocess
REPO = "https://github.com/BilalAjmad/open-kitsurf"
BRANCH = "smallm-agentic-lm"
if not os.path.isdir("open-kitsurf"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
os.chdir("open-kitsurf")
os.environ["PYTHONPATH"] = os.getcwd() + "/src"
print(os.getcwd())
!python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# real data: TinyStories (~300MB slice) -> BPE tokenizer -> token corpus
import os
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/tinystories.txt"):
    !curl -sL -r 0-314572799 -o data/raw/tinystories.txt \
      https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt
!PYTHONPATH=src python3 scripts/prepare_data.py --raw data/raw/tinystories.txt \
  --out-dir data/processed --vocab-size 4096 --tokenizer-sample-mb 20 --tokenize-limit-mb 250

In [ ]:
# brand-new architecture, from scratch, on the GPU
# 90M preset (depth 24). Swap to configs/smallm_nova_30m.json for a faster run.
CFG = "configs/smallm_nova_90m.json"
!PYTHONPATH=src python3 scripts/train_real.py --config {CFG} \
  --data-dir data/processed --out checkpoints/nova_90m \
  --steps 12000 --batch-size 32 --seq-len 512 --accum 2 --lr 6e-4 \
  --save-every 500 --snapshot-dir checkpoints/nova_90m_snaps --snapshot-every 1000 \
  --device auto --resume

In [ ]:
# instruction SFT on real human-written Dolly-15k, then agentic SFT
!PYTHONPATH=src python3 scripts/sft_instruct.py --init checkpoints/nova_90m/model.pt \
  --out checkpoints/nova_90m_instruct --steps 2000 --batch-size 16 --lr 1e-4 --resume
!PYTHONPATH=src python3 scripts/sft_agentic.py --init checkpoints/nova_90m_instruct/model.pt \
  --out checkpoints/nova_90m_agent --steps 1200 --batch-size 16 --lr 1e-4 --resume

In [ ]:
# model soup over the pretraining snapshots, then a quick smoke test
!PYTHONPATH=src python3 scripts/make_soup.py --snapshot-dir checkpoints/nova_90m_snaps \
  --last 8 --out checkpoints/nova_90m_soup --threads 4
!PYTHONPATH=src python3 scripts/chat.py --model checkpoints/nova_90m_instruct \
  --prompt "Once upon a time" --max-new 60 --instruct

In [ ]:
# keep the artifacts in /kaggle/working so they survive as notebook output
import shutil
for d in ["nova_90m", "nova_90m_instruct", "nova_90m_agent", "nova_90m_soup"]:
    src = f"checkpoints/{d}"
    if os.path.isdir(src):
        shutil.copytree(src, f"/kaggle/working/{d}", dirs_exist_ok=True)
        print("saved", d)
!ls -la /kaggle/working